In [0]:
GOLD_PATH = "/Volumes/databricks_wrkspce/default/banking_data/migration/gold"

VALIDATION_CONFIG = {

    "dim_customer": {
        "key": "customer_id",
        "required_columns": [
            "customer_id",
            "first_name",
            "last_name",
            "email",
            "city",
            "customer_segment",
            "date_of_birth"
        ],
        "min_records": 1
    },

    "dim_account": {
        "key": "account_id",
        "required_columns": [
            "account_id",
            "customer_id",
            "branch_id",
            "account_open_date"
        ],
        "min_records": 1
    },

    "dim_branch": {
        "key": "branch_id",
        "required_columns": [
            "branch_id",
            "branch_name",
            "city",
            "state",
            "region"
        ],
        "min_records": 1
    },

    "fact_transactions": {
        "key": "customer_id",
        "required_columns": [
            "customer_id",
            "transaction_count",
            "total_transaction_amount",
            "average_transaction_amount",
            "unique_transaction_types"
        ],
        "min_records": 1
    }
}

In [0]:
def load_gold(table_name):

    return (
        spark.read
        .format("delta")
        .load(f"{GOLD_PATH}/{table_name}")
    )

In [0]:
def validate_record_count(df, minimum):

    count = df.count()

    return {
        "actual": count,
        "expected": f">= {minimum}",
        "status": "PASS" if count >= minimum else "FAIL"
    }

In [0]:
def validate_record_count(df, minimum):

    count = df.count()

    return {
        "actual": count,
        "expected": f">= {minimum}",
        "status": "PASS" if count >= minimum else "FAIL"
    }

In [0]:
def validate_duplicates(df, key):

    total = df.count()

    unique = (
        df.select(key)
        .distinct()
        .count()
    )

    duplicates = total - unique

    return {
        "actual": duplicates,
        "expected": 0,
        "status": "PASS" if duplicates == 0 else "FAIL"
    }

In [0]:
def validate_null_keys(df, key):

    nulls = (
        df.filter(
            F.col(key).isNull()
        ).count()
    )

    return {
        "actual": nulls,
        "expected": 0,
        "status": "PASS" if nulls == 0 else "FAIL"
    }

In [0]:
def validate_schema(df, required_columns):

    actual_columns = set(df.columns)

    missing = [
        column
        for column in required_columns
        if column not in actual_columns
    ]

    return {
        "actual": ", ".join(missing) if missing else "None",
        "expected": "None",
        "status": "PASS" if not missing else "FAIL"
    }

In [0]:
from pyspark.sql import functions as F

results = []

for table_name, config in VALIDATION_CONFIG.items():

    df = load_gold(table_name)

    count_result = validate_record_count(
        df,
        config["min_records"]
    )

    duplicate_result = validate_duplicates(
        df,
        config["key"]
    )

    null_result = validate_null_keys(
        df,
        config["key"]
    )

    schema_result = validate_schema(
        df,
        config["required_columns"]
    )

    results.extend([

        {
            "table": table_name,
            "check": "Record Count",
            **count_result
        },

        {
            "table": table_name,
            "check": "Duplicate Key",
            **duplicate_result
        },

        {
            "table": table_name,
            "check": "Null Key",
            **null_result
        },

        {
            "table": table_name,
            "check": "Schema",
            **schema_result
        }
    ])

In [0]:
validation_df = spark.createDataFrame(results)

display(validation_df)

actual,check,expected,status,table
10000,Record Count,>= 1,PASS,dim_customer
0,Duplicate Key,0,PASS,dim_customer
0,Null Key,0,PASS,dim_customer
None,Schema,None,PASS,dim_customer
20000,Record Count,>= 1,PASS,dim_account
0,Duplicate Key,0,PASS,dim_account
0,Null Key,0,PASS,dim_account
None,Schema,None,PASS,dim_account
200,Record Count,>= 1,PASS,dim_branch
0,Duplicate Key,0,PASS,dim_branch
